In [ ]:
import os
import string
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import precision_score, accuracy_score, recall_score, f1_score
import numpy as np
from tqdm import tqdm

from langchain_huggingface import HuggingFaceEmbeddings
tqdm.pandas()

# Loading dataset

In [10]:
# Diretório dos arquivos
diretorio = "inflação/data/COPOM-Inflation/2_sentences/"

dados = []

for nome_arquivo in os.listdir(diretorio):
    caminho = os.path.join(diretorio, nome_arquivo)
    if os.path.isfile(caminho):
        data = nome_arquivo.replace(".txt", "")  # remove extensão se houver
        with open(caminho, "r", encoding="utf-8") as f:
            for linha in f:
                sentenca = linha.strip()
                if sentenca:
                    dados.append({"data": data, "sentenca": sentenca})


df = pd.DataFrame(dados)
df["data"] = df["data"].str.split("_").str[-1]
df["data"] = pd.to_datetime(df["data"], format="%d%m%Y", errors="coerce")
print(df.head())

        data                                           sentenca
0 2022-05-04  Os membros do Copom analisaram a evolução rece...
1 2022-05-04  As pressões inflacionárias decorrentes da recu...
2 2022-05-04  A reorganização das cadeias de produção globai...
3 2022-05-04  Na visão do Comitê, esses desenvolvimentos pod...
4 2022-05-04  Bancos centrais de países desenvolvidos e emer...


In [11]:
df

,data,sentenca
0,2022-05-04,Os membros do Copom analisaram a evolução rece...
1,2022-05-04,As pressões inflacionárias decorrentes da recu...
2,2022-05-04,A reorganização das cadeias de produção globai...
3,2022-05-04,"Na visão do Comitê, esses desenvolvimentos pod..."
4,2022-05-04,Bancos centrais de países desenvolvidos e emer...
...,...,...
24584,2017-09-06,O Comitê entende que a convergência da inflaçã...
24585,2017-09-06,"O Comitê enfatiza que o processo de reformas, ..."
24586,2017-09-06,"Para a próxima reunião, caso o cenário básico ..."
24587,2017-09-06,"Não obstante as perspectivas acima, o Copom re..."


In [20]:
# Gerar embeddings usando modelo HuggingFace
embeddings = HuggingFaceEmbeddings(model_name="Qwen/Qwen3-Embedding-0.6B")

In [21]:
# Gera o vetor da palavra "inflação"
vetor_inflacao = embeddings.embed_query("inflação")
#vetor_inflacao_alta = embeddings.embed_query("alta na inflação")
#vetor_inflacao_baixa = embeddings.embed_query("baixa na inflação")

In [22]:
# Função para calcular a distância (cosine similarity invertida)
def calcular_distancia(sentenca, vetor_referencia):
    vetor_sentenca = embeddings.embed_query(sentenca)
    # Cosine similarity retorna valor entre -1 e 1; quanto mais próximo de 1, mais similar
    similaridade = cosine_similarity([vetor_sentenca], [vetor_referencia])[0][0]
    # Para transformar em "distância", usamos 1 - similaridade
    distancia = 1 - similaridade
    return distancia

In [23]:
# from df select 100 random samples
df_sample = df.sample(n=100, random_state=42)

In [24]:
df_sample

,data,sentenca
963,2007-06-06,"Em relação à política fiscal, as projeções lev..."
1114,2019-05-08,"Os membros do Copom debateram, também, outros ..."
11432,2002-01-23,"A produção industrial, retirados os efeitos sa..."
24014,2000-06-20,"As operações com cartão de crédito caíram 2, 3..."
7046,2009-10-21,A contribuição da absorção doméstica para o cr...
...,...,...
9539,2003-07-23,Tiveram influência para a queda a maior oferta...
10378,2001-09-19,projeta-se uma inclinação para a curva de juro...
4794,2005-11-23,"Por sua vez, o crescimento do IPA-DI deveu-se ..."
4203,2001-05-23,A elevação da meta para a Taxa Selic nas duas ...


In [25]:
df_sample["inflation"] = df_sample["sentenca"].progress_apply(lambda x: calcular_distancia(x, vetor_inflacao))

100%|██████████| 100/100 [00:01<00:00, 58.19it/s]


In [26]:
df_sample

,data,sentenca,inflation
963,2007-06-06,"Em relação à política fiscal, as projeções lev...",0.612389
1114,2019-05-08,"Os membros do Copom debateram, também, outros ...",0.658990
11432,2002-01-23,"A produção industrial, retirados os efeitos sa...",0.620387
24014,2000-06-20,"As operações com cartão de crédito caíram 2, 3...",0.658797
7046,2009-10-21,A contribuição da absorção doméstica para o cr...,0.644740
...,...,...,...
9539,2003-07-23,Tiveram influência para a queda a maior oferta...,0.629942
10378,2001-09-19,projeta-se uma inclinação para a curva de juro...,0.494648
4794,2005-11-23,"Por sua vez, o crescimento do IPA-DI deveu-se ...",0.547560
4203,2001-05-23,A elevação da meta para a Taxa Selic nas duas ...,0.590733


In [ ]:
# avaliação manual das sentenças
for index, row in df_sample.iterrows():
    sentenca = row['sentenca']
    related = row.get('related', None)
    if related is not None:
        continue
    resposta = input(f"A sentença a seguir é sobre inflação? (y/n): {sentenca}\n")
    while resposta.lower() not in ['y', 'n']:
        print("Resposta inválida. Por favor, responda com 'y' para sim ou 'n' para não.")
        resposta = input(f"A sentença a seguir é sobre inflação? (y/n): {sentenca}\n")
    df_sample.at[index, 'related'] = resposta.lower()

In [32]:
df_sample

,data,sentenca,inflation,related
963,2007-06-06,"Em relação à política fiscal, as projeções lev...",0.612389,n
1114,2019-05-08,"Os membros do Copom debateram, também, outros ...",0.658990,n
11432,2002-01-23,"A produção industrial, retirados os efeitos sa...",0.620387,n
24014,2000-06-20,"As operações com cartão de crédito caíram 2, 3...",0.658797,n
7046,2009-10-21,A contribuição da absorção doméstica para o cr...,0.644740,n
...,...,...,...,...
9539,2003-07-23,Tiveram influência para a queda a maior oferta...,0.629942,y
10378,2001-09-19,projeta-se uma inclinação para a curva de juro...,0.494648,n
4794,2005-11-23,"Por sua vez, o crescimento do IPA-DI deveu-se ...",0.547560,y
4203,2001-05-23,A elevação da meta para a Taxa Selic nas duas ...,0.590733,n


In [ ]:
for i in range(1, 10):
    threshold = i / 10
    df_sample[f'Grade_{i:02d}'] = df_sample['inflation'].apply(lambda x: 'y' if x < threshold else 'n')

In [ ]:
df_sample

: 

In [40]:
df_sample.to_csv('inflação/data/sample_sentences_qwen_06b.csv', index=False)

In [ ]:
for i in range(1, 10):
    grade_col = f'Grade_{i:02d}'
    precision = precision_score(df_sample['related'], df_sample[grade_col], pos_label='y')
    accuracy = accuracy_score(df_sample['related'], df_sample[grade_col])
    recall = recall_score(df_sample['related'], df_sample[grade_col], pos_label='y')
    f1 = f1_score(df_sample['related'], df_sample[grade_col], pos_label='y')
    
    print(f"Resultados para {grade_col}:")
    print(f"Precisão: {precision:.4f}")
    #print(f"Acurácia: {accuracy:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}\n")

Resultados para Grade_01:
Precisão: 0.0000
Recall: 0.0000
F1 Score: 0.0000

Resultados para Grade_02:
Precisão: 0.0000
Recall: 0.0000
F1 Score: 0.0000

Resultados para Grade_03:
Precisão: 0.0000
Recall: 0.0000
F1 Score: 0.0000

Resultados para Grade_04:
Precisão: 0.0000
Recall: 0.0000
F1 Score: 0.0000

Resultados para Grade_05:
Precisão: 0.6667
Recall: 0.0526
F1 Score: 0.0976

Resultados para Grade_06:
Precisão: 0.6774
Recall: 0.5526
F1 Score: 0.6087

Resultados para Grade_07:
Precisão: 0.4022
Recall: 0.9737
F1 Score: 0.5692

Resultados para Grade_08:
Precisão: 0.3838
Recall: 1.0000
F1 Score: 0.5547

Resultados para Grade_09:
Precisão: 0.3800
Recall: 1.0000
F1 Score: 0.5507



/home/users/u7272942/inflação/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/users/u7272942/inflação/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/users/u7272942/inflação/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result